# 💬 Cookbook 4: Conversations and Memory

Learn how to manage multi-turn conversations and agent memory!

**Topics covered:**
1. Conversation history
2. Conversation managers
3. Session persistence
4. Building chatbots

**Prerequisites**: Complete Cookbooks 1-3 first!

---

In [ ]:
!pip install strands-agents strands-agents-tools -q
print("✅ Ready to go!")

## 🧠 Understanding Conversation Memory

By default, Strands agents **remember the conversation** within a session. This means:

- Each message builds on previous context
- The agent can reference earlier parts of the conversation
- You can have natural, multi-turn dialogues

## 💬 Step 1: Basic Multi-turn Conversation

In [ ]:
from strands import Agent

# Create an agent
agent = Agent()

# First message
print("User: My name is Alice.")
response1 = agent("My name is Alice.")

print("\n" + "-"*40 + "\n")

# Second message - the agent remembers!
print("User: What's my name?")
response2 = agent("What's my name?")

print("\n" + "="*50)
print("✅ The agent remembered your name from the previous message!")

## 📜 Step 2: Accessing Conversation History

You can inspect the conversation history stored in the agent.

In [ ]:
from strands import Agent

agent = Agent()

# Have a conversation
agent("What is 2 + 2?")
agent("And what if I multiply that by 3?")

# Access the message history
print("📜 Conversation History:")
print(f"   Total messages: {len(agent.messages)}")

for i, msg in enumerate(agent.messages):
    role = msg.get('role', 'unknown')
    content = msg.get('content', [])
    
    # Extract text content
    text = ""
    if isinstance(content, list):
        for item in content:
            if isinstance(item, dict) and 'text' in item:
                text = item['text'][:100]  # Truncate for display
                break
    
    print(f"\n   [{i}] {role}: {text}...")

## 🔄 Step 3: Resetting Conversation

Sometimes you want to start fresh. You can clear the conversation history.

In [ ]:
from strands import Agent

agent = Agent()

# First conversation
print("User: My favorite color is blue.")
agent("My favorite color is blue.")

print("\n--- Resetting conversation ---\n")

# Clear messages to start fresh
agent.messages.clear()

# Now the agent won't remember
print("User: What's my favorite color?")
response = agent("What's my favorite color?")

print("\n" + "="*50)
print("✅ The agent doesn't remember after reset!")

## 📏 Step 4: Sliding Window Conversation Manager

For long conversations, you can use a sliding window to manage context length.

In [ ]:
from strands import Agent
from strands.agent.conversation_manager import SlidingWindowConversationManager

# Create a conversation manager that keeps last 10 messages
conversation_manager = SlidingWindowConversationManager(
    window_size=10  # Keep only the last 10 messages
)

# Create agent with the conversation manager
agent = Agent(conversation_manager=conversation_manager)

# Simulate a longer conversation
print("Having a conversation with sliding window...\n")

for i in range(5):
    response = agent(f"This is message number {i + 1}. Please acknowledge.")
    print(f"Message {i + 1} sent. Total messages in history: {len(agent.messages)}")

print("\n" + "="*50)
print("✅ The sliding window keeps only recent messages!")

## 🤖 Step 5: Building a Simple Chatbot

Let's build an interactive chatbot that remembers context!

In [ ]:
from strands import Agent
from strands_tools import calculator, current_time

def create_chatbot():
    """Create a chatbot with a friendly personality."""
    return Agent(
        system_prompt="""You are a friendly and helpful assistant named Buddy.
        You remember everything from our conversation.
        Keep your responses concise but warm.
        You can help with math and tell the time.""",
        tools=[calculator, current_time]
    )

def chat_session():
    """Run an interactive chat session."""
    chatbot = create_chatbot()
    
    print("🤖 Buddy the Chatbot")
    print("=" * 40)
    print("Type 'quit' to exit, 'reset' to start fresh")
    print("=" * 40 + "\n")
    
    # Simulate a few turns (in real use, this would be a loop)
    test_messages = [
        "Hi! My name is Sam.",
        "What time is it?",
        "What's 15% of 85?",
        "What's my name again?"
    ]
    
    for user_input in test_messages:
        print(f"You: {user_input}")
        response = chatbot(user_input)
        print("\n" + "-"*40 + "\n")

# Run the chat session
chat_session()

## 💾 Step 6: File-Based Session Persistence

Save conversations to disk so they persist between runs.

In [ ]:
from strands import Agent
from strands.session import FileSessionManager
import os

# Create a session manager that saves to a file
session_manager = FileSessionManager(
    session_id="my_chat_session",
    storage_dir="./sessions"  # Directory to store sessions
)

# Create agent with session persistence
agent = Agent(session_manager=session_manager)

print("Session 1: First conversation")
response = agent("Remember this: The secret code is 42.")

print("\n" + "="*50)
print(f"Session saved! Check ./sessions/ for the file.")

# In a real scenario, you could restart Python and restore:
# session_manager2 = FileSessionManager(session_id="my_chat_session", storage_dir="./sessions")
# agent2 = Agent(session_manager=session_manager2)
# agent2("What was the secret code?")  # Would remember!

## 🎯 Step 7: Context-Aware Assistant

Let's build a more sophisticated assistant that uses context effectively.

In [ ]:
from strands import Agent, tool

# Create tools that work with user preferences
@tool
def set_user_preference(key: str, value: str) -> str:
    """Store a user preference.
    
    Args:
        key: The preference name (e.g., 'name', 'favorite_color').
        value: The preference value.
    
    Returns:
        Confirmation message.
    """
    # In real use, this would store to a database
    return f"✅ Stored preference: {key} = {value}"

@tool
def get_recommendations(category: str) -> list:
    """Get personalized recommendations.
    
    Args:
        category: Category of recommendations (books, movies, music).
    
    Returns:
        List of recommendations.
    """
    recommendations = {
        "books": ["The Pragmatic Programmer", "Clean Code", "Design Patterns"],
        "movies": ["Inception", "The Matrix", "Interstellar"],
        "music": ["Classical Focus", "Lo-Fi Beats", "Jazz for Coding"]
    }
    return recommendations.get(category.lower(), ["No recommendations found"])

# Create a personalized assistant
assistant = Agent(
    system_prompt="""You are a personal assistant that remembers user preferences.
    Use the tools to store and recall preferences.
    Always be helpful and personalized in your responses.""",
    tools=[set_user_preference, get_recommendations]
)

# Have a personalized conversation
print("User: I love science fiction movies.")
response = assistant("I love science fiction movies. Please remember that and give me some movie recommendations.")

print("\n" + "-"*40 + "\n")

print("User: What kind of movies did I say I like?")
response = assistant("What kind of movies did I say I like?")

## 🏋️ Exercise: Build Your Own Chatbot

Create a specialized chatbot for a specific use case!

In [ ]:
from strands import Agent, tool

# 🏋️ YOUR EXERCISE: Build a specialized chatbot!
# Ideas:
# - A cooking assistant that remembers dietary preferences
# - A language tutor that tracks progress
# - A task manager that remembers your to-do list

# Example structure:
@tool
def your_custom_tool(param: str) -> str:
    """Your tool description."""
    return "Your result"

# your_chatbot = Agent(
#     system_prompt="Your specialized prompt here...",
#     tools=[your_custom_tool]
# )

# Have a conversation:
# response = your_chatbot("Your first message")
# response = your_chatbot("Your follow-up message")

print("💡 Create your own specialized chatbot!")

## ✅ Summary

You've learned:

1. **Conversation memory** - Agents remember context by default
2. **Message history** - Access and inspect conversation history
3. **Resetting conversations** - Start fresh when needed
4. **Sliding window** - Manage long conversations efficiently
5. **Session persistence** - Save conversations to disk
6. **Building chatbots** - Create interactive assistants

---

## 🚀 Next Steps

Continue to **Cookbook 5: Multi-Agent Systems** to learn about coordinating multiple agents!

---

## 📚 Resources

- [Conversation Management](https://strandsagents.com/latest/user-guide/concepts/agents/conversation-management/)
- [Session Management](https://strandsagents.com/latest/user-guide/concepts/agents/session-management/)